In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
import copy
import matplotlib.pyplot as plt
from tqdm import tqdm
from imblearn.over_sampling import SMOTE

In [ ]:
combined_df_FE = pd.read_csv(r'..\data\processed\combined_df_FE.csv')

X = combined_df_FE.drop(['class'],axis=1)
y = combined_df_FE['class']


smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
CONFIG = {
    "batch_size" : 4096,
    "epochs": 150,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "patience": 15,
    "n_splits": 5,
    "random_state": 42,
    "num_workers": 0,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray = None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

In [ ]:
class StellarNet(nn.Module):

    def __init__(self, input_dim: int, num_classes: int = 3, hidden_dims: list = [512, 512, 256, 256, 128], dropout: float = 0.2):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.blocks = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            in_d  = hidden_dims[i]
            out_d = hidden_dims[i + 1]
            block = nn.Sequential(
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(out_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            shortcut = (
                nn.Linear(in_d, out_d, bias=False)
                if in_d != out_d else nn.Identity()
            )
            self.blocks.append(nn.ModuleDict({
                "block": block,
                "shortcut": shortcut,
            }))

        self.head = nn.Linear(hidden_dims[-1], num_classes)

    def forward(self, x):
        x = self.input_proj(x)
        for m in self.blocks:
            residual = m["shortcut"](x)
            x = m["block"](x) + residual
        return self.head(x)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for X_batch, y_batch in tqdm(loader):
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=1)
        total_correct += (preds == y_batch).sum().item()
        total_loss += loss.item() * len(y_batch)
        total+= len(y_batch)

    return total_loss / total, total_correct / total

In [ ]:
@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_preds = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda"):
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

        preds = logits.argmax(dim=1)
        total_correct += (preds == y_batch).sum().item()
        total_loss += loss.item() * len(y_batch)
        total += len(y_batch)
        all_preds.extend(preds.cpu().numpy())

    return total_loss / total, total_correct / total, np.array(all_preds)

In [ ]:
def train_pipeline_nn(X:pd.DataFrame, y:pd.Series, config:dict = CONFIG):

    X_arr = X.values.astype(np.float32)
    y_arr = y.values.astype(np.int64)

    input_dim   = X_arr.shape[1]
    num_classes = len(np.unique(y_arr))

    cv = StratifiedKFold(
        n_splits=config["n_splits"],
        shuffle=True,
        random_state=config["random_state"],)